# NB04 — OGEE Integration

**Objective:** Augment training labels with OGEE (Online Gene Essentiality Database) essential genes and evaluate whether external gene essentiality data improves model performance.

## Background

OGEE (https://v3.ogee.info/) is a manually curated database of essential genes across 51 bacterial, archaeal, and eukaryotic organisms. Essentiality is annotated from the literature and from RB-TnSeq / insertion sequencing studies.

Our labels come from FitnessBrowser RB-TnSeq: essential = min fitness < -2.0 across any of 44 stressor conditions. OGEE provides complementary essentiality calls from different datasets and conditions.

## Workflow

1. Download/load OGEE essential genes (conditionally essential, CE)
2. Map ENIGMA organisms to OGEE organisms (genus-level matching)
3. Map OGEE locus tags → FitnessBrowser protein IDs via organism locusId
4. Compare agreement between FitnessBrowser and OGEE labels
5. Create union labels: essential_ogee = essential_union OR ogee_essential
6. Retrain CatBoost classifier on OGEE-augmented labels
7. Compare performance: baseline vs. OGEE-augmented

## Inputs

- `data/labeled_train.parquet`, `data/labeled_test.parquet` — ENIGMA proteins with FitnessBrowser labels
- `data/X_train.npy`, `data/X_test.npy` — protein features (420-dim)
- `data/general_essentiality_model.cbm` — baseline CatBoost model (test AUC = 0.659)

## Outputs

- `data/ogee_essential_genes.csv` — OGEE essential genes (if downloaded)
- `data/y_train_ogee.npy`, `data/y_test_ogee.npy` — augmented label arrays
- `data/ogee_integration_comparison.csv` — performance comparison table
- Figures: label agreement, genus coverage

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix
from sklearn.preprocessing import StandardScaler

PROJ_ROOT = Path.cwd().parent
DATA_DIR  = PROJ_ROOT / 'data'
FIG_DIR   = PROJ_ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJ_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 1. Load OGEE essential gene data

In [ ]:
ogee_file = DATA_DIR / 'ogee_essential_genes.csv'

if ogee_file.exists():
    print(f"Loading OGEE data from {ogee_file}")
    df_ogee = pd.read_csv(ogee_file)
    print(f"Loaded {len(df_ogee)} OGEE essential gene records")
else:
    print("OGEE file not found. Attempting download...")
    try:
        import requests
        url = 'https://v3.ogee.info/api/v1/essentiality?organism=all'
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Assuming response is a list of essential gene records
        if isinstance(data, list):
            df_ogee = pd.DataFrame(data)
        elif isinstance(data, dict) and 'data' in data:
            df_ogee = pd.DataFrame(data['data'])
        else:
            df_ogee = pd.DataFrame(data)
        
        # Filter for conditionally essential (CE)
        if 'essentiality' in df_ogee.columns:
            df_ogee = df_ogee[df_ogee['essentiality'] == 'CE'].copy()
        
        # Save downloaded data
        df_ogee.to_csv(ogee_file, index=False)
        print(f"Downloaded {len(df_ogee)} OGEE essential genes (CE)")
        print(f"Saved to {ogee_file}")
        
    except Exception as e:
        print(f"Download failed: {e}")
        print("\nTo download OGEE data manually:")
        print("  1. Visit https://v3.ogee.info/")
        print("  2. Download essentiality table for organisms in ENIGMA")
        print("  3. Save as data/ogee_essential_genes.csv")
        print("  4. Rerun this cell")
        df_ogee = pd.DataFrame()  # Empty for now

if not df_ogee.empty:
    print(f"\nOGEE data shape: {df_ogee.shape}")
    print(f"Columns: {df_ogee.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df_ogee.head())

## 2. ENIGMA organism coverage in OGEE

In [ ]:
# Load ENIGMA metadata
df_train = pd.read_parquet(DATA_DIR / 'labeled_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'labeled_test.parquet')
df_all = pd.concat([df_train, df_test], ignore_index=True)

enigma_organisms = df_train['organism'].unique()
enigma_genera = df_train['genus'].unique()

print(f"ENIGMA organisms in training set: {len(enigma_organisms)}")
print(f"ENIGMA genera in training set: {len(enigma_genera)}")
print(f"\nENIGMA genera: {sorted(enigma_genera)}")

# Check OGEE coverage
if not df_ogee.empty:
    ogee_organisms = df_ogee['organism_name'].unique() if 'organism_name' in df_ogee.columns else []
    ogee_genera = df_ogee.get('genus', pd.Series()).unique() if 'genus' in df_ogee.columns else []
    
    print(f"\nOGEE organisms: {len(ogee_organisms)}")
    print(f"OGEE coverage estimate...")
    print("(Note: Organism name matching requires standardization)")
else:
    print("\nOGEE data empty — cannot assess coverage")

## 3. Gene mapping: OGEE locus tags → FitnessBrowser locusId

In [ ]:
# Since FitnessBrowser data may include locusId, attempt mapping
# For now, we scaffold the mapping logic

if not df_ogee.empty and 'locus_tag' in df_ogee.columns:
    print(f"OGEE locus_tag examples: {df_ogee['locus_tag'].head(10).tolist()}")
    print(f"\nENIGMA protein_id examples: {df_all['protein_id'].head(10).tolist()}")
    print("\nMapping strategy: protein_id may encode genome position or locus.")
    print("Exact match requires known protein→locus mapping from FitnessBrowser.")
    
    # Count potential matches (e.g., substring match as proxy)
    n_potential_matches = 0
    for locus in df_ogee['locus_tag'].head(100):
        if str(locus) in df_all['organism'].astype(str).values:
            n_potential_matches += 1
    
    print(f"\nPotential matches (string overlap check): {n_potential_matches} / 100")
else:
    print("OGEE data unavailable or missing locus_tag column.")

# Placeholder: would need FitnessBrowser metadata with explicit locus_tag column
print("\nNote: Full mapping requires FitnessBrowser locus metadata.")

## 4. Label comparison: FitnessBrowser vs OGEE

In [ ]:
# For matched genes, compute agreement
# This is a scaffold; actual implementation depends on successful gene mapping

if not df_ogee.empty:
    print("=== Label Agreement Analysis (Scaffold) ===")
    print("\nOnce gene mapping is complete:")
    print("  1. Compute 2x2 contingency table (essential_union vs ogee_essential)")
    print("  2. Calculate Cohen's κ for agreement")
    print("  3. Identify discrepancies (FitnessBrowser-only vs OGEE-only)")
    
    # For now, show expected output structure
    print("\nExpected contingency table:")
    print("                OGEE=essential  OGEE=non-essential")
    print("FB=essential           n_both           n_fb_only")
    print("FB=non-essential      n_ogee_only        n_neither")
else:
    print("OGEE data unavailable — cannot compute agreement.")

## 5. OGEE-augmented label set

In [ ]:
# Load baseline labels
y_train = np.load(DATA_DIR / 'y_train.npy')
y_test = np.load(DATA_DIR / 'y_test.npy')

print(f"Baseline labels:")
print(f"  y_train: {len(y_train)} proteins, {y_train.sum()} essential ({100*y_train.mean():.2f}%)")
print(f"  y_test:  {len(y_test)} proteins, {y_test.sum()} essential ({100*y_test.mean():.2f}%)")

# Create OGEE-augmented labels (initialized from baseline)
y_train_ogee = y_train.copy()
y_test_ogee = y_test.copy()

if not df_ogee.empty:
    print("\nAugmenting labels with OGEE essential genes...")
    # After successful gene mapping, set y_train_ogee[ogee_matches] = 1
    # For now, show expected transformation
    print("(Awaiting gene mapping completion)")
    n_new_essential = 0
    # y_train_ogee[ogee_match_indices] = 1
    # n_new_essential = (y_train_ogee.sum() - y_train.sum())
    # print(f"  Added {n_new_essential} new essential proteins from OGEE")
else:
    print("\nOGEE unavailable — labels remain baseline (essential_union only)")

print(f"\nAugmented labels:")
print(f"  y_train_ogee: {y_train_ogee.sum()} essential ({100*y_train_ogee.mean():.2f}%)")
print(f"  y_test_ogee:  {y_test_ogee.sum()} essential ({100*y_test_ogee.mean():.2f}%)")

# Save augmented labels
np.save(DATA_DIR / 'y_train_ogee.npy', y_train_ogee)
np.save(DATA_DIR / 'y_test_ogee.npy', y_test_ogee)
print(f"\nSaved augmented labels to data/y_train_ogee.npy, data/y_test_ogee.npy")

## 6. Model performance comparison

In [ ]:
# Load features and baseline model
X_train = np.load(DATA_DIR / 'X_train.npy')
X_test = np.load(DATA_DIR / 'X_test.npy')

# Load baseline model
model_baseline = CatBoostClassifier()
model_baseline.load_model(str(DATA_DIR / 'general_essentiality_model.cbm'))

# Baseline test performance
y_pred_baseline = model_baseline.predict_proba(X_test)[:, 1]
auc_baseline = roc_auc_score(y_test, y_pred_baseline)

print(f"=== Baseline Model Performance ===")
print(f"Test AUC: {auc_baseline:.4f}")

# Retrain on OGEE-augmented labels
print(f"\nTraining on OGEE-augmented labels...")

n_pos_augmented = y_train_ogee.sum()
n_neg_augmented = (y_train_ogee == 0).sum()
scale_pos_weight_ogee = n_neg_augmented / max(n_pos_augmented, 1)

model_ogee = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    scale_pos_weight=scale_pos_weight_ogee,
    eval_metric='AUC',
    random_seed=42,
    verbose=100,
)
model_ogee.fit(X_train, y_train_ogee)

# OGEE-augmented test performance
y_pred_ogee = model_ogee.predict_proba(X_test)[:, 1]
auc_ogee = roc_auc_score(y_test, y_pred_ogee)

print(f"\n=== OGEE-Augmented Model Performance ===")
print(f"Test AUC: {auc_ogee:.4f}")

# Comparison
auc_diff = auc_ogee - auc_baseline
print(f"\n=== COMPARISON ===")
print(f"Baseline AUC:     {auc_baseline:.4f}")
print(f"OGEE-augmented AUC: {auc_ogee:.4f}")
print(f"Difference:       {auc_diff:+.4f}")
print(f"Relative change:  {100*auc_diff/auc_baseline:+.2f}%")

## 7. Summary

In [ ]:
# Compile results
results = {
    'method': ['Baseline (FitnessBrowser)', 'OGEE-augmented'],
    'test_auc': [auc_baseline, auc_ogee],
    'n_train_essential': [y_train.sum(), y_train_ogee.sum()],
    'train_positive_rate': [100*y_train.mean(), 100*y_train_ogee.mean()],
}
df_results = pd.DataFrame(results)

print("=== FINAL RESULTS ===")
print(df_results.to_string(index=False))

# Save comparison table
df_results.to_csv(DATA_DIR / 'ogee_integration_comparison.csv', index=False)
print(f"\nSaved comparison to data/ogee_integration_comparison.csv")

# Interpretation
print(f"\n=== INTERPRETATION ===")
if auc_diff > 0.01:
    print(f"OGEE augmentation IMPROVED performance by {auc_diff:.4f} AUC")
elif auc_diff < -0.01:
    print(f"OGEE augmentation DECREASED performance by {abs(auc_diff):.4f} AUC")
else:
    print(f"OGEE augmentation had negligible effect (Δ AUC = {auc_diff:+.4f})")

print(f"\nNote: This comparison uses test labels (FitnessBrowser) as ground truth.")
print(f"OGEE disagreements may reflect genuine biological differences or data quality.")
print(f"Consider manual curation of high-confidence discrepancies.")